# ChainLens Phase 6 — Baseline and Ablation

This notebook has two exhibits:

1. A shallow decision tree as a **feature-importance exhibit**, fitted without the test-split address. It is not a performance claim.
2. A per-rule **ablation detection test** against the committed baseline snapshot. It proves that removing a rule removes exactly that rule's contribution and changes nothing outside that rule's hit set.

The locked rule config is never weakened: each ablation uses a frozen in-memory copy with one weight set to zero. The live `config/rules.yaml` and `risk_scores` table are not mutated.


## 0. Imports and provenance

The tree and ablation read the current locked baseline. The test-split address is excluded from the tree's fitting universe.


In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier, export_text
from sqlalchemy import text

from chainlens.ablation import (
    assert_baseline_restored,
    load_baseline,
    run_ablation,
)
from chainlens.db import get_pg_engine
from chainlens.features import get_scaled_feature_columns

pd.set_option("display.max_columns", 50)
engine = get_pg_engine()

scaled_columns, scaler_version = get_scaled_feature_columns(engine)
with engine.connect() as conn:
    features = pd.read_sql(text("SELECT * FROM address_features"), conn)
    labels = pd.read_sql(text("SELECT * FROM labels"), conn)
    splits = pd.read_sql(text("SELECT * FROM label_splits"), conn)
    scores = pd.read_sql(text("SELECT * FROM risk_scores"), conn)

baseline = load_baseline()
print(f"scaler_version: {scaler_version}")
print(f"scaled features ({len(scaled_columns)}): {scaled_columns}")
print(f"baseline rows: {len(baseline)}")
print(f"baseline rules_version: {baseline['rules_version'].iloc[0]}")


## 1. Decision-tree exhibit (train-only fit)

The split table only labels three feature-bearing addresses: two train and one test. A literal two-row tree would be unreadable,
so the fitting universe is **every non-test feature address**. Train-split labels are positives; every other non-test address is
an unlabeled negative. The one test-split address (`KAGGLE_FRAUD`) is excluded from `X` and `y` entirely.

This is a deliberate, documented choice: it avoids test-label leakage while giving the tree enough rows to expose feature ordering.
The test split is **not** scored or reported.


In [ ]:
test_addresses = set(splits.query("split == 'test'")["address"].astype(str).str.lower())
train_label_addresses = set(
    labels.merge(splits.query("split == 'train'"), on="address", how="inner")["address"]
    .astype(str)
    .str.lower()
)
train_universe = features[~features["address"].isin(test_addresses)].copy()
train_universe["flagged"] = train_universe["address"].isin(train_label_addresses).astype(int)

X_train = train_universe[scaled_columns].fillna(0.0)
y_train = train_universe["flagged"].to_numpy()

print(f"fit rows: {len(train_universe)}")
print(f"positives: {int(y_train.sum())}; negatives: {int((y_train == 0).sum())}")
print(f"test-split addresses in split table: {len(test_addresses)}")
print(f"feature rows removed: {len(features) - len(train_universe)}")


In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42, class_weight="balanced")
tree.fit(X_train, y_train)

importance = (
    pd.DataFrame({"feature": scaled_columns, "tree_importance": tree.feature_importances_})
    .sort_values("tree_importance", ascending=False)
    .reset_index(drop=True)
)
rule_weights = pd.DataFrame(
    [
        {"rule": "R1", "name": "circular_flow", "weight": 25},
        {"rule": "R2", "name": "fanout", "weight": 20},
        {"rule": "R3", "name": "sanctioned_1hop", "weight": 30},
        {"rule": "R4", "name": "pagerank_top5pct", "weight": 15},
        {"rule": "R5", "name": "apriori_match", "weight": 20},
    ]
)
rule_weights["normalized_weight"] = rule_weights["weight"] / rule_weights["weight"].sum()
print("Tree feature importances:")
display(importance)
print("Hand-set rule weights (normalized):")
display(rule_weights)


The tree can only see the five numeric/scaled transaction features. Four of the five rules depend on graph or pattern inputs
(`cycle_member`, `sanctioned_1hop`, `pagerank_top5`, `apriori_match`) that are not in this matrix. The tree's top features are
therefore a proxy for the fan-out/value family, not a like-for-like replacement for the rules. This is the interpretability gap.


In [ ]:
print(export_text(tree, feature_names=scaled_columns, max_depth=3))


### Train confusion matrix

This is **on train, overfit by construction**. It measures whether the tree can learn the labels, not whether it generalizes.
No test metric is computed or reported (`# Not reported: n=1 test positive.`).


In [ ]:
train_prediction = tree.predict(X_train)
cm = confusion_matrix(y_train, train_prediction, labels=[0, 1])
display(
    pd.DataFrame(
        cm,
        index=["actual negative", "actual positive"],
        columns=["predicted negative", "predicted positive"],
    )
)
# Not reported: n=1 test positive. No test-set scoring call exists in this notebook.


## 2. Ablation detection test

Target set is the Phase 6 table: R1 → the three cycle wallets, R2 → `FANOUT_WHALE`, R3 → `SANCTIONED_1HOP`, R4 → the top-ranked
`pagerank_top5` wallet in the top 10, and R5 → untestable on the fixture. A global rule removal changes every wallet that hit the
rule; `other_wallets_changed` therefore checks wallets **outside the rule's hit set**. All are `False`.

`run_ablation` is strict: an unexpected target delta or a changed outside wallet raises. The `finally` block verifies the current
score/trace content hash still matches the baseline. The live rules file hash is also checked inside `run_ablation`.


In [ ]:
report = None
details = None
try:
    report, details = run_ablation(baseline=baseline)
    display(
        report[
            [
                "rule_id",
                "target",
                "baseline_raw",
                "ablated_raw",
                "raw_delta",
                "expected_weight",
                "affected_wallets",
                "other_wallets_changed",
                "isolated",
                "testable",
                "note",
            ]
        ]
    )
    display(pd.DataFrame([details["exchange_hot_r2"]]))
finally:
    restored_hash = assert_baseline_restored(baseline=baseline)
    print("BASELINE RESTORE OK")
    print(f"content_sha256: {restored_hash}")


### `EXCHANGE_HOT` R2 ablation arithmetic

`EXCHANGE_HOT` is not the primary R2 target (`FANOUT_WHALE` is), but it is in R2's affected set because it also fired R2.
Its arithmetic is the important downweight check:

- baseline raw = 35; adjusted = 35 × 0.3 = 10.5; final = 10
- ablated raw = 15; adjusted = (35 − 20) × 0.3 = 4.5; final = 4
- final delta = **6**, not the raw 20

The notebook does not report a metric delta from ablation. On n=0 real positives, that would be meaningless.


In [ ]:
assert report is not None
testable = report[report["testable"]]
assert testable["other_wallets_changed"].eq(False).all()
assert testable["isolated"].all()
assert report.loc[report["rule_id"] == "R5", "testable"].eq(False).all()
print("All testable rules isolated; no outside wallet changed.")
print("R5 is untestable on fixture: zero Apriori flags; covered by tests/test_apriori.py.")


## 3. Interpretation

The rules encode a specific ordering (sanctioned proximity → circular flow → fan-out → PageRank) chosen for interpretability,
not learned from the data. The decision tree learns a similar feature set but distributes weight differently, which is the
trade-off the project measures. Ablation confirms each rule is isolated: removing any rule removes exactly its contribution from
exactly the wallets that hit it, with no outside changes. The one fixture test label is insufficient to validate the rules
end-to-end; the fixture tests validate rule mechanics, and real-data validation is deferred.
